In [1]:
# importing packages and modules
import numpy as np
import matplotlib as mpl
import scipy.stats as stats
from scipy.optimize import minimize, check_grad
from io_utils import *
from utils import *
from plotting_utils import *
from analysis_utils import *
import dynamic_glmhmm
from scipy.stats import multivariate_normal, norm
sns.set_context("talk")

colormap = ['tab:purple','tab:pink','tab:cyan','yellowgreen', 'olive']
colorsStates = ['tab:orange','tab:blue','tab:green','tab:purple', 'tab:brown']
myFeatures = [['bias','stimulus', 'previous choice', 'previous reward'],['bias','contrast left','contrast right', 'previous choice', 'previous reward']]
ibl_data_path = '../data_IBL'
# dfAll = pd.read_csv(ibl_data_path + '/Ibl_processed.csv')
dfAll = pd.read_csv(ibl_data_path + '/IBL_processed_extra.csv')

labChosen =  ['angelakilab','churchlandlab','wittenlab']
subjectsAll = []
for lab in labChosen:
    subjects = np.unique(dfAll[dfAll['lab'] == lab]['subject']).tolist()
    subjectsAll = subjectsAll + subjects

# missing data
if ('NYU-01' in subjectsAll):
    subjectsAll.remove('NYU-01')
if ('NYU-06' in subjectsAll):
    subjectsAll.remove('NYU-06')
if ('CSHL_007' in subjectsAll):
    subjectsAll.remove('CSHL_007')
if ('CSHL049' in subjectsAll):
    subjectsAll.remove('CSHL049')
if ('CSHL024' in subjectsAll):
    subjectsAll.remove('CSHL024')


In [2]:
dfAll.head()

,contrastLeft,contrastRight,choice,feedbackType,probabilityLeft,lab,subject,date,session,correctSide,response_times,RT
0,1.0,0.0,1.0,0.0,1.0,angelakilab,IBL-T1,2019-02-09,1.0,0.0,NaN,NaN
1,1.0,0.0,1.0,0.0,1.0,angelakilab,IBL-T1,2019-02-09,1.0,0.0,NaN,NaN
2,0.0,1.0,1.0,1.0,1.0,angelakilab,IBL-T1,2019-02-09,1.0,1.0,NaN,NaN
3,0.5,0.0,0.0,1.0,0.5,angelakilab,IBL-T1,2019-02-09,1.0,0.0,NaN,NaN
4,0.5,0.0,0.0,1.0,0.9,angelakilab,IBL-T1,2019-02-09,1.0,0.0,NaN,NaN


# Simulations to show number of trials needed to recover good weights

In [34]:
sessions = [25, 50]
trials = [100,200,400,800]
Nsamples = 20

K = 3
D = 2
pTanh = 5
signedStimulus = True
sessStop = None

avg_model = np.load(f'../data_IBL/average_animals_fig4-5_best_parameters_dynamic.npz')
dynamicW = avg_model['bestAvgW']
dynamicP = avg_model['bestAvgP']
dynamicpi = np.ones((K))/K
truepi = np.ones((K))/K

subject = 'ibl_witten_15'
x, y, sessInd_old, correctSide, responseTimes = get_mouse_design(dfAll, subject, sessStop=None, signedStimulus=signedStimulus, pTanh=pTanh)
biasedBlockTrials, biasedBlockStartInd, biasedBlockSession, firstBlockSession = get_design_biased_blocks(dfAll, subject, sessInd_old, sessStop)
N = x.shape[0]
sess = len(sessInd_old)-1
x = x[:,:2] # only keeping bias and stimulus




In [39]:


standard_model = np.load(f'../data_IBL/all_animals/Best_allAnimals_standardGLMHMM_{K}-state_pTanh={pTanh}_signedStimulus={signedStimulus}.npz')
standardP = standard_model['P'][:50]
standardW = standard_model['W'][:50,:,:2]

print(dynamicW.shape)
print(standardW.shape)
print(standardP.shape)

(50, 3, 2, 2)
(50, 3, 2, 2)
(50, 3, 3)


In [40]:

print(dynamicP[0].sum(axis=1))
print(standardP[0].sum(axis=1))

# normalizing transition  matrix
for s in range(50):
    for j in range(K):
        dynamicP[s,j,:] = dynamicP[s,j,:] / dynamicP[s,j,:].sum()
        standardP[s,j,:] = standardP[s,j,:] / standardP[s,j,:].sum()

print(dynamicP[0].sum(axis=1))
print(standardP[0].sum(axis=1))

[1. 1. 1.]
[0.99999992 0.99999957 0.99999945]
[1. 1. 1.]
[1. 1. 1.]


In [46]:
sigmaList = [10**x for x in list(np.arange(-3,1,0.5,dtype=float))] + [10**x for x in list(np.arange(1,4,1,dtype=float))]
bestSigmaInd = 8 
bestSigma = sigmaList[bestSigmaInd-1]
alphaList = [2*(10**x) for x in list(np.arange(-1,6,0.5,dtype=float))]
bestAlphaInd = 2  # Choosing best sigma index across animals
bestAlpha = alphaList[bestAlphaInd]
maxiter = 250

# sessions = [25]
# trials = [100]
# Nsamples = 1
sessions = [25, 50]
trials = [100,200,400,800]
Nsamples = 20



for Nsess in sessions:
    print(f'session {Nsess}')
    for Ntrial in trials:
        print(f'Ntrial {Ntrial}')
        truex = np.ones((Nsess * Ntrial, 2))
        sessInd= [i * Ntrial for i in range(Nsess+1)]
        for s in range(Nsess):
            truex[Ntrial * s : Ntrial * s + Ntrial, 1] = np.random.choice(x[sessInd_old[s]:sessInd_old[s+1],1], size=Ntrial, replace=True).flatten()
        
        N = truex.shape[0]
        dGLM_HMM = dynamic_glmhmm.dynamic_GLMHMM(N,K,D,2)
        trueW, trueP = reshape_parameters_session_to_trials(dynamicW[:Nsess+1], dynamicP[:Nsess+1], sessInd)
        initW, initP = reshape_parameters_session_to_trials(standardW[:Nsess+1], standardP[:Nsess+1], sessInd)
        for i in range(Nsamples):    
            print(f'sample = {i}')
            truey, truez = dGLM_HMM.simulate_data_given_x(truex, trueW, trueP, truepi, sessInd, seed=i)
            presentAll = np.ones((N))
            fitP, _, fitW, trainLl = dGLM_HMM.fit(truex, truey, presentAll, initP, truepi, initW, sigma=reshapeSigma(bestSigma, K, D), alpha=bestAlpha, A=standardP[0], sessInd=sessInd, maxIter=maxiter, tol=1e-3, model_type='dynamic',  L2penaltyW=0, priorDirP = None, fit_init_states=False)
            np.savez(f'../simulations/dynamicGLMHMM_simulation_NatComms_sample={i}_Nsess={Nsess}_Ntrial={Ntrial}', P=fitP, allW=fitW, y=truey, z=truez, x=truex, trueW=trueW, trueP=trueP)


session 25
Ntrial 100
sample = 0
0
sample = 1
0
sample = 2
0
sample = 3
0
sample = 4
0
sample = 5
0
sample = 6
0
sample = 7
0
sample = 8
0
sample = 9
0
sample = 10
0
sample = 11
0
sample = 12
0
sample = 13
0
sample = 14
0
sample = 15
0
sample = 16
0
sample = 17
0
sample = 18
0
sample = 19
0
Ntrial 200
sample = 0
0
sample = 1
0
sample = 2
0
sample = 3
0
sample = 4
0
sample = 5
0
sample = 6
0
sample = 7
0
sample = 8
0
sample = 9
0
sample = 10
0
sample = 11
0
sample = 12
0
sample = 13
0
sample = 14
0
sample = 15
0
sample = 16
0
sample = 17
0
sample = 18
0
sample = 19
0
Ntrial 400
sample = 0
0
sample = 1
0
sample = 2
0
sample = 3
0
sample = 4
0
sample = 5
0
sample = 6
0
sample = 7
0
sample = 8
0
sample = 9
0
sample = 10
0
sample = 11
0
sample = 12
0
sample = 13
0
sample = 14
0
sample = 15
0
sample = 16
0
sample = 17
0
sample = 18
0
sample = 19
0
Ntrial 800
sample = 0
0
sample = 1
0
sample = 2
0
sample = 3
0
sample = 4
0
sample = 5
0
sample = 6
0
sample = 7
0
sample = 8
0
sample = 9
0
sampl